# Tipos de capas de una Red Recurrente (RNN)


Para cada capa veremos **qué hace**, **qué parámetros** recibe y **cuándo usarla**.

> Plantilla mental de una RNN:
> `Input → Embedding → [ RNN layer ] → Dense de salida`

## Preparación

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

VOCAB = 5000   # vocabulario
MAX_LEN = 100  # longitud de secuencia
# datos de juguete
X = np.random.randint(0, VOCAB, (200, MAX_LEN))
y = np.random.randint(0, 2, 200)
print('Datos de ejemplo:', X.shape)

Datos de ejemplo: (200, 100)


---
## 1. `Embedding` — De palabras a vectores

**Qué hace:** convierte cada índice de palabra (un entero) en un vector
denso de números. Palabras similares acaban con vectores parecidos.
Es la primera capa en cualquier modelo de texto.

```python
layers.Embedding(input_dim, output_dim, input_length)
```

| Parámetro | Qué significa | Valores típicos |
|-----------|---------------|----------------|
| `input_dim` | Tamaño del vocabulario | 5000, 10000, 20000 |
| `output_dim` | Dimensión del vector por palabra | 32, 64, 128 |
| `input_length` | Longitud de las secuencias | depende de tus datos |

**Cuándo usarla:** siempre que la entrada sean palabras (índices).
No la necesitas si tus datos ya son numéricos (series de tiempo).

**El `output_dim`:** un embedding de 64 significa que cada palabra se
representa con 64 números. Más dimensiones capturan más matices,
pero tardan más.

In [ ]:
emb = layers.Embedding(input_dim=VOCAB, output_dim=64, input_length=MAX_LEN)


ejemplo = np.array([[1, 42, 300]])
print('Ejemplo: ', ejemplo)
salida = emb(ejemplo)
print('Entrada:', ejemplo.shape, '→ Salida:', salida.shape)
print('Salida: ', salida)

---
## 2. `SimpleRNN` — La RNN básica

**Qué hace:** lee la secuencia paso a paso manteniendo un estado oculto.
Es la más sencilla pero olvida rápido en secuencias largas.

```python
layers.SimpleRNN(units, return_sequences=False, activation='tanh')
```

| Parámetro | Qué significa | Valores típicos |
|-----------|---------------|----------------|
| `units` | Dimensión del estado oculto | 32, 64, 128 |
| `return_sequences` | `False`: solo da la salida final. `True`: da una salida por cada paso | `False` si es la última capa RNN; `True` si hay otra RNN después |
| `activation` | Activación del estado | `'tanh'` (por defecto) |
| `dropout` | Dropout en las entradas | 0.0 a 0.5 |
| `recurrent_dropout` | Dropout en el estado oculto | 0.0 a 0.5 |

**Cuándo usarla:** solo para pruebas rápidas o secuencias muy cortas (<50 tokens).

In [ ]:
m1 = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB, 64),
    layers.SimpleRNN(32),     
    layers.Dense(1, activation='sigmoid')
])
m1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
m1.summary()
print('\nParámetros de SimpleRNN:', m1.layers[1].count_params())

---
## 3. `LSTM` — Memoria a largo plazo

**Qué hace:** añade una celda de memoria y tres puertas (olvido, entrada, salida)
que controlan qué información mantener. Soluciona el problema de olvidar
en secuencias largas.

```python
layers.LSTM(units, return_sequences=False, dropout=0.0, recurrent_dropout=0.0)
```

| Parámetro | Qué significa | Valores típicos |
|-----------|---------------|----------------|
| `units` | Dimensión del estado oculto | 64, 128, 256 |
| `return_sequences` | Igual que SimpleRNN | `False` (última) / `True` (apilada) |
| `dropout` | Dropout en entradas | 0.0 a 0.3 |
| `recurrent_dropout` | Dropout en estados | 0.0 a 0.3 |

**Cuándo usarla:** es la opción por defecto para casi cualquier problema
de secuencias. Úsala siempre que dudes.

**Nota sobre `return_sequences`:** si apilas dos LSTM, la primera necesita
`return_sequences=True` para pasar toda la secuencia a la siguiente.

In [ ]:
m2 = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB, 64),
    layers.LSTM(64),                   # la más usada
    layers.Dense(1, activation='sigmoid')
])
m2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
m2.summary()
print('\nUn LSTM tiene ~4x más parámetros que un SimpleRNN del mismo tamaño')
print('SimpleRNN(32):', m1.layers[1].count_params(), 'params')
print('LSTM(64):     ', m2.layers[1].count_params(), 'params')

---
## 4. `GRU` — LSTM simplificado

**Qué hace:** similar al LSTM pero con solo 2 puertas (reset y update)
en vez de 3. Es más rápido y muchas veces obtiene resultados igual de buenos.

```python
layers.GRU(units, return_sequences=False, dropout=0.0, recurrent_dropout=0.0)
```

| Parámetro | Qué significa | Valores típicos |
|-----------|---------------|----------------|
| `units` | Dimensión del estado oculto | 64, 128 |
| `return_sequences` | Igual que LSTM | `False` / `True` |
| `dropout` / `recurrent_dropout` | Regularización | 0.0 a 0.3 |

**Cuándo usarla:** cuando quieres algo más rápido que LSTM sin perder
mucha calidad. Buen primer intento.

In [ ]:
m3 = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB, 64),
    layers.GRU(64),
    layers.Dense(1, activation='sigmoid')
])
m3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
m3.summary()
print('\nGRU tiene ~3x parámetros de SimpleRNN (vs ~4x del LSTM)')

---
## 5. `Bidirectional` — Leer en ambos sentidos

**Qué hace:** envuelve cualquier capa RNN (SimpleRNN, LSTM o GRU) y la
ejecuta dos veces: una leyendo de izquierda a derecha y otra de derecha
a izquierda. Combina ambas salidas.

```python
layers.Bidirectional(layers.LSTM(64))
```

| Parámetro | Qué significa | Valores típicos |
|-----------|---------------|----------------|
| `layer` | La capa RNN que quieres hacer bidireccional | `layers.LSTM(64)`, `layers.GRU(32)` |
| `merge_mode` | Cómo combinar ida y vuelta | `'concat'` (por defecto) |

**Cuándo usarla:** cuando la tarea permite ver toda la secuencia antes
de decidir (clasificación, NER). No sirve para generación de texto
(donde no puedes ver el futuro).

**Nota:** con `merge_mode='concat'` (por defecto), la salida tiene el
**doble** de unidades (64 forward + 64 backward = 128).

In [ ]:
m4 = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB, 64),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(1, activation='sigmoid')
])
m4.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
m4.summary()
print('\nCon Bidirectional, la salida del LSTM(64) pasa a 128 (64+64)')

---
## 6. Apilar capas RNN (`return_sequences`)

**Qué hace:** puedes poner varias capas RNN una encima de otra para que
el modelo sea más profundo. Para eso, las capas intermedias necesitan
devolver la secuencia completa (`return_sequences=True`).

```
Embedding → LSTM(return_sequences=True) → LSTM() → Dense
```

**Regla:** la última capa RNN lleva `return_sequences=False` (o no lo pones,
que es el valor por defecto). Las anteriores llevan `True`.

In [ ]:
m5 = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB, 64),
    layers.LSTM(64, return_sequences=True),    
    layers.LSTM(32),                         
    layers.Dense(1, activation='sigmoid')
])
m5.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
m5.summary()
print('\nLa primera LSTM pasa 100 vectores de 64 a la segunda LSTM')
print('La segunda LSTM resume todo en un vector de 32')

---
## Tabla resumen: ¿cuándo uso cada capa?

| Capa | Para qué sirve | ¿Cuándo la pongo? |
|------|----------------|-------------------|
| **Embedding** | Convierte palabras en vectores | Primera capa si la entrada es texto |
| **SimpleRNN** | RNN básica, olvida rápido | Solo pruebas rápidas / secuencias cortas |
| **LSTM** | Memoria a largo plazo (3 puertas) | Opción por defecto en casi todo |
| **GRU** | Versión rápida de LSTM (2 puertas) | Cuando quieres velocidad |
| **Bidirectional** | Lee ida y vuelta | Clasificación de texto (no generación) |
| **Dense** | Clasifica con lo que la RNN extrajo | Siempre al final |

### Plantilla mental

**Texto** (clasificación):
```
Input → Embedding → LSTM (o GRU) → Dense(sigmoid/softmax)
```

### Parámetro clave: `return_sequences`

| Situación | Valor |
|-----------|-------|
| Última (o única) capa RNN | `False` (por defecto) |
| Hay otra RNN después | `True` |
| Quieres una salida por cada paso | `True` |